# Exon Inclusion Analysis

In [ ]:
!pip install ipykernel
!python -m ipykernel install --user --name finalproject --display-name "Python (final project)"

Installed kernelspec finalproject in /home/biouser/.local/share/jupyter/kernels/finalproject


In [ ]:
## 0. Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from scipy.stats import spearmanr, bootstrap
from gtfparse import read_gtf
import pysam  
from matplotlib.colors import LinearSegmentedColormap
from functions import (
    load_type_neurons,
    compute_spearman_corr,
    clean_names,
    spearman_corr_labels,
)

## 1. Load Data
- Load `altExonUsage_devel_type.gz`
- Filter only column with `Neuron` 
- Combine `Exon` and `Gene` to `GE` (`Exon::Gene`)

In [ ]:
type_neurons = load_type_neurons("../../data/altExonUsage_devel_type.gz")
type_neurons.head(20)

,P14_Hippocampus_ExciteNeuron,P14_Hippocampus_InhibNeuron,P14_VisCortex_ExciteNeuron,P14_VisCortex_InhibNeuron,P21_Hippocampus_ExciteNeuron,P21_Hippocampus_InhibNeuron,P21_VisCortex_ExciteNeuron,P21_VisCortex_InhibNeuron,P28_Hippocampus_ExciteNeuron,P28_Hippocampus_InhibNeuron,P28_VisCortex_ExciteNeuron,P28_VisCortex_InhibNeuron,P56_Hippocampus_ExciteNeuron,P56_Hippocampus_InhibNeuron,P56_VisCortex_ExciteNeuron,P56_VisCortex_InhibNeuron
GE,,,,,,,,,,,,,,,,
chr1_162273620_162273649_-::ENSMUSG00000040265.16,0.032520,0.000000,0.008152,0.000000,0.049351,0.024000,0.000000,0.000000,0.015544,0.007812,0.001163,0.000000,0.015831,0.026316,0.005181,0.001821
chr1_186690721_186690804_-::ENSMUSG00000039239.14,0.000000,NaN,NaN,NaN,0.075472,NaN,NaN,NaN,0.083333,NaN,NaN,NaN,0.139073,NaN,0.019608,0.125000
chr1_172286198_172286396_-::ENSMUSG00000007097.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.818182,NaN
chr1_83384894_83384972_-::ENSMUSG00000026163.17,0.608364,0.642140,0.707604,0.657718,0.631307,0.707113,0.767779,0.776471,0.670919,0.666667,0.789824,0.805970,0.641826,0.694915,0.781662,0.712150
chr1_172279313_172279467_-::ENSMUSG00000007097.14,1.000000,NaN,1.000000,NaN,0.823529,NaN,0.823529,NaN,1.000000,NaN,1.000000,NaN,1.000000,NaN,0.965517,NaN
chr1_163253977_163254048_-::ENSMUSG00000026586.16,0.545455,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chr1_172287200_172287468_-::ENSMUSG00000007097.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
chr1_16486020_16486150_-::ENSMUSG00000025920.19,0.204724,0.032967,0.240229,0.097222,0.222116,0.230769,0.275465,0.102564,0.223958,0.000000,0.230177,0.132075,0.218418,0.050633,0.220895,0.145867
chr1_162675605_162675841_-::ENSMUSG00000040225.15,0.812834,0.714286,0.824022,0.848485,0.891247,0.795455,0.818493,0.914286,0.869822,1.000000,0.841040,NaN,0.950530,0.840000,0.900850,0.959184


## 2. Performn pairwise Correlation

#### Process data for Spearman correlation
- Remove rows with missing data `NaN`
- Compute Spearman correlation matrix

In [ ]:
corr = compute_spearman_corr(type_neurons)
CN = clean_names(corr)
CN

,P14_HIPP_Excite,P14_HIPP_Inhib,P14_VIS_Excite,P14_VIS_Inhib,P21_HIPP_Excite,P21_HIPP_Inhib,P21_VIS_Excite,P21_VIS_Inhib,P28_HIPP_Excite,P28_HIPP_Inhib,P28_VIS_Excite,P28_VIS_Inhib,P56_HIPP_Excite,P56_HIPP_Inhib,P56_VIS_Excite,P56_VIS_Inhib
P14_HIPP_Excite,1.000000,0.865320,0.909911,0.886845,0.853609,0.841284,0.821143,0.845471,0.849383,0.841355,0.851881,0.817556,0.891090,0.885976,0.828041,0.816164
P14_HIPP_Inhib,0.865320,1.000000,0.852621,0.899736,0.843048,0.884450,0.814208,0.847335,0.830215,0.866666,0.819943,0.838889,0.836201,0.896754,0.813307,0.869893
P14_VIS_Excite,0.909911,0.852621,1.000000,0.910102,0.830537,0.830371,0.810601,0.866398,0.828769,0.833890,0.855997,0.821240,0.882145,0.870460,0.833206,0.823089
P14_VIS_Inhib,0.886845,0.899736,0.910102,1.000000,0.877425,0.881019,0.870836,0.886242,0.856335,0.864176,0.869230,0.900208,0.861776,0.893214,0.864582,0.898332
P21_HIPP_Excite,0.853609,0.843048,0.830537,0.877425,1.000000,0.861511,0.813749,0.879860,0.766540,0.811592,0.726914,0.761101,0.821703,0.885722,0.807427,0.833759
P21_HIPP_Inhib,0.841284,0.884450,0.830371,0.881019,0.861511,1.000000,0.844983,0.901320,0.825075,0.878993,0.811337,0.875430,0.825536,0.896380,0.837813,0.890832
P21_VIS_Excite,0.821143,0.814208,0.810601,0.870836,0.813749,0.844983,1.000000,0.889425,0.728493,0.789257,0.724341,0.785212,0.787906,0.869884,0.830261,0.819163
P21_VIS_Inhib,0.845471,0.847335,0.866398,0.886242,0.879860,0.901320,0.889425,1.000000,0.849540,0.847341,0.861349,0.902924,0.855445,0.882308,0.880106,0.913320
P28_HIPP_Excite,0.849383,0.830215,0.828769,0.856335,0.766540,0.825075,0.728493,0.849540,1.000000,0.840163,0.814149,0.816738,0.863503,0.884956,0.749237,0.770761
P28_HIPP_Inhib,0.841355,0.866666,0.833890,0.864176,0.811592,0.878993,0.789257,0.847341,0.840163,1.000000,0.827930,0.871407,0.805134,0.885399,0.803124,0.851962


In [ ]:
f4a_labels = spearman_corr_labels(CN)

NameError: name 'cnn' is not defined